# CAFE — black-box evaluation (real LLMs)

Hand CAFE your system as a **black box**, vary some factors, `evaluate()`, read the
attribution. Models are **LiteLLM** strings, so any provider works:
`"ollama/gpt-oss:120b"`, `"gpt-4o"`, `"anthropic/claude-..."`. Keys come from a
local `.env`.

> *Black-box mode* — CAFE sees only inputs and outputs. *Technique mode*
> (registering pipeline components) is a separate notebook, not built yet.

In [1]:
import os

import cafe
from cafe._env import load_env

load_env()  # loads OLLAMA_API_KEY (and any other provider keys) from the nearest .env
print("cafe", cafe.__version__, "| OLLAMA_API_KEY loaded:", bool(os.environ.get("OLLAMA_API_KEY")))

cafe 0.0.1 | OLLAMA_API_KEY loaded: True


## 1. Define the system (the black box)

Just `run(config, item) -> output`. It reads the chosen levels from `config` and
calls whatever model. `cafe.complete(model, messages)` is a thin LiteLLM helper —
use your own client here if you prefer.

In [2]:
PROMPTS = {
    "concise": "Answer correctly and concisely.",
    "cot": "Reason step by step, then end with a line 'Answer: <final answer>'.",
}

async def qa_system(config, item):
    answer = await cafe.complete(
        config["model"],
        messages=[
            {"role": "system", "content": PROMPTS[config["prompt"]]},
            {"role": "user", "content": item["text"]},
        ],
        temperature=float(config.get("temperature", 0.0)),
    )
    return {"output": answer}

# Each item carries the question text and a reference answer for the judge.
QUESTIONS = [
    {"id": "bat_ball", "text": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?", "reference": "$0.05"},
    {"id": "train", "text": "A train travels 60 km in the first hour, then 80 km/h for the next 2 hours. How far has it traveled in total?", "reference": "220 km"},
    {"id": "prime8", "text": "What is the 8th prime number?", "reference": "19"},
]

## 2. Build the study

The `Study` holds the system, the **factors**, the **dataset**, the **rubric**, and
the **judge**.

- **`replications`** — how many times each (config × item) is run, to measure the
  system's run-to-run noise (`judge_replications` does the same for the judge).
- The **rubric** declares its `scale_type` (`ordinal` / `numeric` / `binary`) so the
  statistics treat it correctly. It's fully customizable.
- The **judge** is just a LiteLLM model string; its prompt is a research-grounded
  preset (MT-Bench / Inspect), overridable per rubric.

In [3]:
from cafe import Factor, Level, LLMJudge, Rubric, ScaleType, Study

rubric = Rubric(
    name="correctness_1_5",
    scale_type=ScaleType.ordinal,
    instruction="Judge whether the ANSWER correctly answers the QUESTION, using the REFERENCE as ground truth.",
    levels=[
        Level(1, "wrong", "Incorrect or irrelevant."),
        Level(2, "weak", "Mostly wrong."),
        Level(3, "partial", "Partially correct."),
        Level(4, "good", "Correct, minor issues."),
        Level(5, "perfect", "Fully correct and clear."),
    ],
)

study = Study(
    name="qa-ollama",
    system=qa_system,
    factors=[
        Factor("model", ["ollama/gpt-oss:120b", "ollama/gpt-oss:20b"]),
        Factor("prompt", ["concise", "cot"]),
    ],
    dataset=QUESTIONS,
    rubric=rubric,
    judge=LLMJudge(model="ollama/gpt-oss:120b"),  # any LiteLLM model string; preset="reference_guided"
    replications=1,
)
print(cafe.size(study), "configs ×", len(QUESTIONS), "questions")

4 configs × 3 questions


## 3. See exactly what the judge will be asked

No tokens spent — print the precise prompt for an example answer. Edit
`rubric.instruction`, the levels, or `rubric.prompt_template` to change it.

In [4]:
print(study.preview_judge_prompt("The ball costs 10 cents."))  # a deliberately wrong answer

Please act as an impartial judge. Judge whether the ANSWER correctly answers the QUESTION, using the REFERENCE as ground truth.

[BEGIN DATA]
***
[Question]: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?
***
[Answer]: The ball costs 10 cents.
***
[Reference answer]: $0.05
***
[END DATA]

Begin your evaluation by reasoning step by step: compare the Answer against the
Reference answer, identify and correct any mistakes, and note unsupported claims.
Judge correctness and helpfulness — not style, length, or formatting.

Score on this scale:
  1 = wrong: Incorrect or irrelevant.
  2 = weak: Mostly wrong.
  3 = partial: Partially correct.
  4 = good: Correct, minor issues.
  5 = perfect: Fully correct and clear.

Then finish with exactly one final line, and nothing after it:
GRADE: <integer 1-5>


## 4. Preflight

One question through every configuration — no replication, no judging — to eyeball
the answers and estimate cost before the full study.

In [5]:
pf = await cafe.preflight(study, concurrency=4)
print(pf.show())

preflight — one input through every configuration:
  [model=ollama/gpt-oss:120b·prompt=concise] The ball costs **$0.05**.
  [model=ollama/gpt-oss:20b·prompt=concise] The ball costs **$0.05** (5 cents).
  [model=ollama/gpt-oss:120b·prompt=cot] Let the cost of the ball be \(x\) dollars.   Then the bat costs \(x + 1.00\) dol
  [model=ollama/gpt-oss:20b·prompt=cot] Let    - \(x\) = price of the ball   - \(x + \$1.00\) = price of the bat    The 

full study: 12 cells; est. compute ~57.25s, est. cost n/a (system reports no cost_usd)


## 5. Evaluate (the complete pipeline)

Generate answers → judge → attribute, with progress bars. (In a notebook use
`await cafe.evaluate(...)`; in a script `study.evaluate()` is the sync version.)

In [6]:
result = await cafe.evaluate(study, concurrency=4)
result

/home/fabian/Desktop/PHD/CAFE/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
judging: 100%|██████████| 12/12 [00:07<00:00,  1.67it/s]


Evaluation(12 answers · 4 configs · 3 inputs · 12 ratings · best: model=ollama/gpt-oss:120b·prompt=concise)

### The answers

Just display the object — columns are your factor names, then the `output`.

In [7]:
result.answers

model,prompt,input_id,rep,output,elapsed_s,error
ollama/gpt-oss:120b,concise,prime8,0,The 8th prime number is **19**.,1.639340,None
ollama/gpt-oss:120b,concise,bat_ball,0,The ball costs **$0.05**.,1.720208,None
ollama/gpt-oss:120b,concise,train,0,The train travels \n\n- First hour: \(60 \text{ km}\) \n- Next 2 hours: \(80 \text{ km/h} \times 2 \text{ h} = 160 \text{ km}\)\n\nTotal distance \(= 60 + 160 = \boxed{220 \text{ km}}\).,1.772256,None
ollama/gpt-oss:120b,cot,bat_ball,0,Let the cost of the ball be \(x\) dollars. \nThen the bat costs \(x + 1.00\) dollars (since it is \$1.00 more than the ball).\n\nThe total cost is given as \$1.10:\n\n\[\nx + (x + 1.00) = 1.10\n\]\n\nCombine like terms:\n\n\[\n2x + 1.00 = 1.10\n\]\n\nSubtract \$1.00 from both sides:\n\n\[\n2x = 0.10\n\]\n\nDivide by 2:\n\n\[\nx = 0.05\n\]\n\nSo the ball costs **\$0.05** (5 cents).\n\nAnswer: 0.05 dollars.,3.903060,None
ollama/gpt-oss:20b,concise,bat_ball,0,The ball costs **$0.05** (5 cents).,3.734746,None
ollama/gpt-oss:120b,cot,prime8,0,"The prime numbers in order are: 2, 3, 5, 7, 11, 13, 17, 19, … \nCounting them, the 8th prime is **19**.\n\nAnswer: 19",4.083956,None
ollama/gpt-oss:120b,cot,train,0,The train travels:\n\n- First hour: \(60 \text{ km}\)\n- Next two hours: \(80 \text{ km/h} \times 2 \text{ h} = 160 \text{ km}\)\n\nTotal distance = \(60 \text{ km} + 160 \text{ km} = 220 \text{ km}\).\n\n**Answer: 220 km**,5.774717,None
ollama/gpt-oss:20b,concise,prime8,0,The 8th prime number is **19**.,3.153110,None
ollama/gpt-oss:20b,cot,bat_ball,0,Let \n\n- \(x\) = price of the ball \n- \(x + \$1.00\) = price of the bat (since the bat is \$1.00 more expensive)\n\nThe total cost is \$1.10:\n\n\[\nx + (x + 1.00) = 1.10\n\]\n\nCombine like terms:\n\n\[\n2x + 1.00 = 1.10\n\]\n\nSubtract \$1.00 from both sides:\n\n\[\n2x = 0.10\n\]\n\nDivide by 2:\n\n\[\nx = 0.05\n\]\n\nSo the ball costs **$0.05 (5 cents)**.,4.040287,None
ollama/gpt-oss:20b,concise,train,0,The train covers \(60\text{ km}\) in the first hour. \nFor the next two hours it travels at \(80\text{ km/h}\):\n\n\[\n\text{Distance} = 80 \text{ km/h} \times 2 \text{ h} = 160 \text{ km}\n\]\n\nTotal distance:\n\n\[\n60 \text{ km} + 160 \text{ km} = 220 \text{ km}\n\]\n\n**Answer: 220 km.**,6.538869,None


### The judge's verdicts

In [8]:
result.ratings

model,prompt,input_id,rep,judge_rep,verdict,reasoning
ollama/gpt-oss:120b,cot,bat_ball,0,0,5,None
ollama/gpt-oss:120b,concise,bat_ball,0,0,5,None
ollama/gpt-oss:120b,concise,prime8,0,0,5,None
ollama/gpt-oss:120b,concise,train,0,0,5,None
ollama/gpt-oss:120b,cot,train,0,0,5,None
ollama/gpt-oss:20b,concise,prime8,0,0,5,None
ollama/gpt-oss:20b,concise,bat_ball,0,0,5,None
ollama/gpt-oss:120b,cot,prime8,0,0,5,None
ollama/gpt-oss:20b,cot,bat_ball,0,0,5,None
ollama/gpt-oss:20b,concise,train,0,0,5,None


### Audit one verdict end-to-end

Every rating stores the exact prompt sent and the raw judge response.

In [9]:
r = result.ratings.items[0]
print("PROMPT SENT TO JUDGE\n" + "=" * 40)
print(r.prompt)
print("\nRAW JUDGE RESPONSE\n" + "=" * 40)
print(r.raw_response)
print("\nPARSED ->", r.value_numeric)

PROMPT SENT TO JUDGE
Please act as an impartial judge. Judge whether the ANSWER correctly answers the QUESTION, using the REFERENCE as ground truth.

[BEGIN DATA]
***
[Question]: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?
***
[Answer]: Let the cost of the ball be \(x\) dollars.  
Then the bat costs \(x + 1.00\) dollars (since it is \$1.00 more than the ball).

The total cost is given as \$1.10:

\[
x + (x + 1.00) = 1.10
\]

Combine like terms:

\[
2x + 1.00 = 1.10
\]

Subtract \$1.00 from both sides:

\[
2x = 0.10
\]

Divide by 2:

\[
x = 0.05
\]

So the ball costs **\$0.05** (5 cents).

Answer: 0.05 dollars.
***
[Reference answer]: $0.05
***
[END DATA]

Begin your evaluation by reasoning step by step: compare the Answer against the
Reference answer, identify and correct any mistakes, and note unsupported claims.
Judge correctness and helpfulness — not style, length, or formatting.

Score on this scale:
  1 = wrong: Incorr

## 6. Attribution

Which configuration scored best, and which factor moved the needle?

In [10]:
print(result.attribution.show())

ratings: 12 (12 usable)   factors: model, prompt

per-configuration mean quality:
  5.00  (n= 3)  model=ollama/gpt-oss:120b·prompt=concise
  5.00  (n= 3)  model=ollama/gpt-oss:120b·prompt=cot
  5.00  (n= 3)  model=ollama/gpt-oss:20b·prompt=concise
  5.00  (n= 3)  model=ollama/gpt-oss:20b·prompt=cot

per-factor marginal means:
  model:
     ollama/gpt-oss:120b mean=5.00  n=6
     ollama/gpt-oss:20b mean=5.00  n=6
  prompt:
     concise          mean=5.00  n=6
     cot              mean=5.00  n=6

best configuration: model=ollama/gpt-oss:120b·prompt=concise  (mean 5.00)


## Notes

- **Judge presets** (`cafe.JUDGE_PRESETS`): `reference_guided` (default),
  `mtbench_single`, `criterion_graded` — adapted from MT-Bench / Inspect AI, with
  provenance in the source. Choose via `LLMJudge(model=..., preset=...)`, or set
  `rubric.prompt_template` for a fully custom prompt.
- **Attribution is the descriptive layer** (means + best config). The inferential
  layer — mixed-effects model + significance + ordinal CLMM — is the next slice.
- If every config scores the same, the questions aren't **discriminating** enough
  (ceiling effect) — use harder inputs where weaker configs fail.
- Raise `replications` / `judge_replications` to measure run-to-run nondeterminism.